In [6]:
from utils import *

print("=== [1] Constructing Random Code Instance ===")
q = 2             # Base field characteristic (F_2)
m = 4             # Field extension degree (F_{2^4})
n = 7             # Total code length 
k = 3             # Dimension of the code
r = 2             # Target rank weight of the error vector

n_minus_k = n - k 
Fq = GF(q)
Fqm = GF(q**m, name='z')
V_Fq = Fqm.vector_space()

print(f"Parameters: q={q}, m={m}, n={n}, k={k}, r={r}\n")

# Generate a random Parity-Check matrix H with full row rank
while True:
    H = matrix(Fqm, n_minus_k, n, [Fqm.random_element() for _ in range(n_minus_k * n)])
    if H.rank() == n_minus_k:
        break

# Generate a secret error vector with exact rank r
field_basis = [Fqm.gen()**i for i in range(m)]
support_basis = random.sample(field_basis, r) # Sample r linearly independent elements

e_list = []
for _ in range(n):
    coeffs = [Fq.random_element() for _ in range(r)]
    entry = sum(int(coeffs[i]) * support_basis[i] for i in range(r))
    e_list.append(entry)
e_secret = vector(Fqm, e_list)

# 1. Construct the explicit m x n coordinate Matrix E over the base field Fq
# Each column of E corresponds to the vector representation of e_secret[j]
E_matrix = matrix(Fq, [V_Fq(el) for el in e_secret]).transpose()
assert E_matrix.rank() == r, "Error vector rank mismatch."

print(f"Generated Secret Vector e:\n{e_secret}")
print(f"Underlying Support Matrix E (dimensions {m}x{n}):\n{E_matrix}\n")

# Compute Syndrome
s = H * e_secret

# =====================================================================
# 2. CHOOSE RANDOM COLUMNS FROM THE MATRIX E TO FORM ERASURES
# =====================================================================
num_erasures = 2
J = sorted(random.sample(range(n), num_erasures))

a_J = []
for j in J:
    # Extract the j-th column from the base field matrix E
    column_vector_Fq = E_matrix.column(j)
    
    # Reconstruct the extension field element from its Fq column coordinates
    field_element_Fqm = sum(column_vector_Fq[i] * field_basis[i] for i in range(m))
    a_J.append(Fqm(field_element_Fqm))

print("=== [2] Extracting Correct Column Erasure Hints from Matrix E ===")
print(f"Selected Column Indices (J): {J}")
for idx, j in enumerate(J):
    print(f"  -> Column {j} from matrix E: {list(E_matrix.column(j))} maps to field element: {a_J[idx]}")
print(f"\nFinal a_J array passed to decoder: {a_J}\n")

# =====================================================================
# 3. RUN DECODER AND VERIFY
# =====================================================================
print("=== [3] Running Full-Entry AP Decoder Module ===")
candidate_list = full_entry_ap_decoder_module(H, s, J, a_J, q, m, n, k)
print(f"Number of candidates returned: {len(candidate_list)}")

# Final Verification
success = False
for idx, candidate in enumerate(candidate_list):
    syndrome_check = (H * candidate == s)
    cand_matrix = matrix(Fq, [V_Fq(el) for el in candidate]).transpose()
    rank_check = (cand_matrix.rank() == r)
    
    if syndrome_check and rank_check:
        print(f"[+] Candidate #{idx+1} passes validation filters.")
        if candidate == e_secret:
            print("[++++] SUCCESS: Reconstructed vector matches the secret error exactly!")
            success = True
            break

if not success:
    print("[-] FAILURE: Reconstructed target vector could not be recovered.")

=== [1] Constructing Random Code Instance ===
Parameters: q=2, m=4, n=7, k=3, r=2

Generated Secret Vector e:
(z + 1, z, z, 0, z, z, z)
Underlying Support Matrix E (dimensions 4x7):
[1 0 0 0 0 0 0]
[1 1 1 0 1 1 1]
[0 0 0 0 0 0 0]
[0 0 0 0 0 0 0]

=== [2] Extracting Correct Column Erasure Hints from Matrix E ===
Selected Column Indices (J): [3, 4]
  -> Column 3 from matrix E: [0, 0, 0, 0] maps to field element: 0
  -> Column 4 from matrix E: [0, 1, 0, 0] maps to field element: z

Final a_J array passed to decoder: [0, z]

=== [3] Running Full-Entry AP Decoder Module ===
Number of candidates returned: 5
[+] Candidate #1 passes validation filters.
[+] Candidate #5 passes validation filters.
[++++] SUCCESS: Reconstructed vector matches the secret error exactly!
